In [1]:
import argparse
from transformers import AutoModel
import torch
import torch.nn as nn
from transformers import set_seed
from data import load_bciciv2a
from engine import eval_model
import types
from modeling_reve import Reve
from einops import rearrange
from tqdm.auto import tqdm

BATCH_SIZE = 64
SEED = 42
LR = 0.001
EPOCHS = 1
set_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Loading REVE
hf_model = AutoModel.from_pretrained("brain-bzh/reve-base", trust_remote_code=True, dtype="auto")
pos_bank = AutoModel.from_pretrained("brain-bzh/reve-positions", trust_remote_code=True, dtype="auto")

# Load Dataset
train_loader, val_loader, test_loader = load_bciciv2a(pos_bank, BATCH_SIZE, seed=SEED)

flash_attn not found, install it with `pip install flash_attn` if you want to use it
flash_attn not found, install it with `pip install flash_attn` if you want to use it


Choosing from all possible events


In [2]:
class RevePT(Reve):
    def __init__(self, config, num_subject=9, num_prompt=10):
        super().__init__(config)
        self.subject_token = nn.ParameterList([
            nn.Parameter(torch.randn(1, num_prompt, self.embed_dim))
            for _ in range(num_subject)
        ])

    def forward_pt(self, eeg, pos, subject_id):
        x = self.forward(eeg, pos, return_output=True)
        print(len(x))
        b, c, s, e = x.shape
        x_flat = rearrange(x, "b c s e -> b (c s) e")
        prompt = self.subject_token[subject_id]
        query_output = prompt.expand(b, -1, -1)
        attention_scores = torch.matmul(query_output, x_flat.transpose(-1, -2)) / (self.embed_dim**0.5)
        attention_weights = torch.softmax(attention_scores, dim=-1)  # (b, num_prompt, c*s)
        x_attended = torch.matmul(attention_weights.transpose(-1, -2), query_output)  # (b, c*s, e)
        x_attended = rearrange(x_attended, "b (c s) e -> b c s e", b=b, c=c, s=s)
        x_attended = self.final_layer(x_attended)
        return x_attended

model = RevePT(hf_model.config, num_subject=9, num_prompt=10)


In [3]:
# Add Classification Head
dim = 22 * 5 * 512
model.final_layer = torch.nn.Sequential(
    torch.nn.Flatten(),
    torch.nn.RMSNorm(dim),
    torch.nn.Dropout(0.1),
    torch.nn.Linear(dim, 4),
)

params = list(model.final_layer.parameters())
params.extend(model.subject_token)

In [4]:
def train_one_epoch(model, optimizer, loader, device):
    criterion = torch.nn.CrossEntropyLoss()
    model.train()
    pbar = tqdm(loader, desc="Training", total=len(loader))

    for batch_data in pbar:
        data, target, pos, subj = (
            batch_data["sample"].to(device, non_blocking=True),
            batch_data["label"].to(device, non_blocking=True),
            batch_data["pos"].to(device, non_blocking=True),
            batch_data["subject_id"].to(device, non_blocking=True),
        )
        optimizer.zero_grad()
        with torch.amp.autocast(dtype=torch.float16, device_type="cuda" if torch.cuda.is_available() else "cpu"):
            output = model.forward_pt(data, pos, subj)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        pbar.set_postfix({"loss": loss.item()})

In [5]:
optimizer = torch.optim.AdamW(params, lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=3)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

best_val_acc = 0
best_final_layer = None

for epoch in range(EPOCHS):
    print(f"Epoch {epoch + 1}/{EPOCHS}")
    train_one_epoch(model, optimizer, train_loader, device)
    b_acc = eval_model(model, val_loader, device)["balanced_acc"]
    if b_acc > best_val_acc:
        best_val_acc = b_acc
        best_final_layer = model.final_layer.state_dict()
    print(f"Validation balanced accuracy: {b_acc:.4f}, best: {best_val_acc:.4f}")
    scheduler.step(b_acc)


model.final_layer.load_state_dict(best_final_layer)
results = eval_model(model, test_loader, device)

# Results
print("acc", results["acc"])
print("balanced_acc", results["balanced_acc"])
print("cohen_kappa", results["cohen_kappa"])
print("f1", results["f1"])
print("auroc", results["auroc"])
print("auc_pr", results["auc_pr"])

Epoch 1/1


Training:   0%|          | 0/57 [00:00<?, ?it/s]

23


AttributeError: 'list' object has no attribute 'shape'